# Patient-Adaptive Interaction-Term Composite

This notebook runs the Patient-Adaptive interaction model separately from the SRM Global Linear notebook. The model allows MRI feature weights to vary with participant demographic/genetic modulators, while clinical scores remain benchmarks only and are not used for training or tuning.

**Validation rule:** split by `subject`, not by `pair_id`, so `V1V2` and `V2V3` intervals from the same participant are never separated across train/test.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import Config, DEFAULT_CONFIG, set_global_seeds
from src.data.audit import modelling_pair_count_table
from src.data.trackfa_pairs import trackfa_pairs_to_long
from src.eval.adaptive_tuning import (
    available_modulator_sets,
    evaluate_patient_adaptive_candidates,
    feature_rank_table,
    selected_feature_sets_from_rank,
)
from src.eval.panel_combinations import evaluate_srm_panel_combinations
from src.features.panels import resolve_a_priori_70_panel
from src.models.interaction import InteractionLinearComposite
from src.reporting.model_performance import assemble_performance_rows, validation_test_gap_table
from src.reporting.fold_comparison import fold_train_test_clinical_benchmark_table

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
long_df = trackfa_pairs_to_long(pairs_df)
panel = resolve_a_priori_70_panel(pairs_df.columns)
if panel.missing_features:
    raise KeyError(f"A priori 70-feature panel has missing columns: {panel.missing_features}")
imaging_cols = [f for f in panel.features if f in long_df.columns]
subject_col = "pair_id"
split_group_col = "subject"
RANDOM_SEED = DEFAULT_CONFIG.random_state
N_BOOT = 200
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} pair-interval visit rows, {len(imaging_cols)} a priori features")
print("Available participant columns:", [c for c in ["age", "sex", "gender", "gaa_1", "gaa_2", "onset_age", "disease_duration", "site"] if c in long_df.columns])


Canonical modelling-cohort pair counts


,count,interval,n,definition
0,N12,V1->V2,108,subjects with a V1V2 annual pair row
1,N23,V2->V3,99,subjects with a V2V3 annual pair row
2,N13,V1->V3,90,subjects with both V1V2 and V2V3 annual pair rows
3,N123,"V1,V2,V3",90,subjects represented across all three visits v...


Loaded trackfa_pairs_drop3poms.csv: 414 pair-interval visit rows, 70 a priori features
Available participant columns: ['age', 'sex', 'gaa_1', 'gaa_2', 'onset_age', 'disease_duration', 'site']


In [2]:
# Fold-level clinical train/test benchmark for supervisor review.
fold_train_test_clinical_benchmarks = fold_train_test_clinical_benchmark_table(
    long_df,
    pairs_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    clinical_scales=("FARS", "SARA"),
    pair_types=("V1V2", "V2V3"),
)
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
fold_train_test_clinical_benchmarks.to_csv(
    RESULTS_DIR / "interaction_term_fold_train_test_clinical_benchmarks.csv",
    index=False,
)
clinical_display_cols = [
    "fold",
    "clinical_scale",
    "train_n_subjects",
    "test_n_subjects",
    "clinical_train_n_pairs",
    "clinical_test_n_pairs",
    "clinical_train_d",
    "clinical_test_d",
    "clinical_train_minus_test_d",
]
print("Interaction/adaptive model fold-level train/test clinical benchmarks")
display(fold_train_test_clinical_benchmarks[clinical_display_cols])


Interaction/adaptive model fold-level train/test clinical benchmarks


,fold,clinical_scale,train_n_subjects,test_n_subjects,clinical_train_n_pairs,clinical_test_n_pairs,clinical_train_d,clinical_test_d,clinical_train_minus_test_d
0,1,FARS,93,24,162,45,0.361604,0.585453,-0.223849
1,1,SARA,93,24,162,45,0.379539,0.503772,-0.124232
2,2,FARS,93,24,159,48,0.384278,0.481659,-0.097381
3,2,SARA,93,24,159,48,0.357044,0.553715,-0.196670
4,3,FARS,94,23,169,38,0.424238,0.332809,0.091429
5,3,SARA,94,23,169,38,0.407784,0.402383,0.005401
6,4,FARS,94,23,167,40,0.406111,0.421720,-0.015609
7,4,SARA,94,23,167,40,0.458035,0.196119,0.261916
8,5,FARS,94,23,171,36,0.460649,0.213516,0.247133
9,5,SARA,94,23,171,36,0.421557,0.332821,0.088736


## 1. Experiment Settings

The Patient-Adaptive model uses all available demographic/genetic modulators in the long-form modelling table. `site` is intentionally not included because it is an acquisition/scanner/site variable, not a participant demographic feature.


In [3]:
# First review the per-feature longitudinal d_z ranking that drives progression-aware mRMR.
a_priori_feature_rank = feature_rank_table(long_df, imaging_cols, subject_col=subject_col, visit_col="visit")
print("A priori feature relevance rank for supervisor review")
display(a_priori_feature_rank)

# Use both anatomical and modality/metric panel families to find strong reduced pools.
anatomical_eval = evaluate_srm_panel_combinations(
    long_df,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=2,
    random_seed=RANDOM_SEED,
    n_boot=100,
    panel_family="anatomical",
)
modality_eval = evaluate_srm_panel_combinations(
    long_df,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=2,
    random_seed=RANDOM_SEED,
    n_boot=100,
    panel_family="modality",
)
panel_pool_summary = pd.concat([anatomical_eval["summary"], modality_eval["summary"]], ignore_index=True)
panel_pool_summary = panel_pool_summary.sort_values(
    ["test_score", "test_annual_interval_gap", "feature_count"],
    ascending=[False, True, True],
    kind="mergesort",
).reset_index(drop=True)
panel_pool_summary["pooled_rank"] = np.arange(1, len(panel_pool_summary) + 1)
print("Best base SRM panel pools to feed adaptive tuning")
display(panel_pool_summary[[
    "pooled_rank", "panel_family", "panel_combo", "feature_count", "validation_score", "test_score", "validation_minus_test",
    "test_dz_v1_v2", "test_dz_v2_v3", "test_annual_interval_gap", "test_p_progression",
]].head(12))


A priori feature relevance rank for supervisor review


,rank_by_abs_mean_annual_dz,feature,dz_v1_v2,dz_v2_v3,mean_annual_dz,abs_mean_annual_dz,annual_interval_gap
0,1,Cerebellum_Cortex_CerebNet,-0.877971,-0.403101,-0.640536,0.640536,0.474871
1,2,Pons,-0.777246,-0.359113,-0.568179,0.568179,0.418134
2,3,Cerebellum_WM_CerebNet,-0.510012,-0.342412,-0.426212,0.426212,0.167600
3,4,TotalBrainGMVol_nocereb,-0.402182,-0.422809,-0.412496,0.412496,0.020628
4,5,Lateral_Ventricle,0.535869,0.286104,0.410986,0.410986,0.249765
...,...,...,...,...,...,...,...
65,66,sFA_c3c5,0.017725,-0.051963,-0.017119,0.017119,0.069689
66,67,FA_SFOF,0.068821,-0.037374,0.015724,0.015724,0.106195
67,68,RD_UNC,-0.016262,-0.011255,-0.013758,0.013758,0.005007
68,69,FA_SLF,-0.050552,0.041930,-0.004311,0.004311,0.092482


Best base SRM panel pools to feed adaptive tuning


,pooled_rank,panel_family,panel_combo,feature_count,validation_score,test_score,validation_minus_test,test_dz_v1_v2,test_dz_v2_v3,test_annual_interval_gap,test_p_progression
0,1,modality,brain_structural+brain_diffusion_rd,40,0.665371,0.808814,-0.143443,1.000076,0.617553,0.382523,0.813131
1,2,modality,brain_structural+brain_diffusion_rd+spinal_dif...,41,0.665801,0.806692,-0.140891,1.004893,0.608490,0.396403,0.823232
2,3,modality,brain_structural+brain_diffusion_rd+spinal_dif...,41,0.673005,0.804808,-0.131803,1.003624,0.605991,0.397633,0.818603
3,4,modality,brain_structural+spinal_structural+brain_diffu...,41,0.643928,0.803921,-0.159993,1.000286,0.607556,0.392730,0.803030
4,5,modality,brain_structural+brain_diffusion_rd+spinal_dif...,42,0.664245,0.803485,-0.139240,0.998894,0.608076,0.390818,0.813552
5,6,modality,brain_structural+spinal_structural+brain_diffu...,42,0.643699,0.801421,-0.157722,1.001957,0.600886,0.401071,0.803030
6,7,modality,brain_structural+spinal_structural+brain_diffu...,42,0.651526,0.798758,-0.147232,0.999659,0.597857,0.401802,0.813552
7,8,modality,brain_structural+spinal_structural+brain_diffu...,43,0.640501,0.797665,-0.157164,0.994857,0.600474,0.394383,0.808081
8,9,anatomical,structural_cerebellum_brainstem+structural_cer...,27,0.638421,0.766322,-0.127901,0.953737,0.578907,0.374830,0.800505
9,10,modality,brain_structural+spinal_diffusion_fa,14,0.694067,0.764460,-0.070393,0.932588,0.596331,0.336257,0.827862


## 2. Patient-Adaptive Tuning

Each candidate is evaluated with participant-grouped folds. The displayed tuning evidence keeps `d12` and `d23` separate and uses mean annual validation `d_z = (d12 + d23) / 2` for performance review. For this exploratory Patient-Adaptive comparison, the final candidate is the numerically best all-demographic-modulator configuration.


In [4]:

# Lock the patient-adaptive screen to the current SRM-selected 16 features.
# This keeps the interaction denominator aligned with the 117-subject / 207-pair
# SRM-clinical benchmark cohort instead of reintroducing broader feature-set screens.
RESULTS_DIR = REPO_ROOT / "results"
selected_feature_path = RESULTS_DIR / "weekly_feature_importance.csv"
if not selected_feature_path.exists():
    raise FileNotFoundError(f"Run the feature-selection notebook first: {selected_feature_path}")
selected_feature_table = pd.read_csv(selected_feature_path)
selected_16_features = selected_feature_table["feature"].astype(str).tolist()
if len(selected_16_features) != 16:
    raise ValueError(f"Expected the elbow-selected 16 features, found {len(selected_16_features)}")
missing_selected = [f for f in selected_16_features if f not in long_df.columns]
if missing_selected:
    raise KeyError(f"Selected features missing from long TRACK-FA table: {missing_selected}")

feature_sets_for_run = {"srm_elbow_selected_k16": selected_16_features}
modulator_sets = available_modulator_sets(long_df.columns)

modulator_denominators = []
for mods in modulator_sets:
    cols = [subject_col, "visit", split_group_col] + selected_16_features + list(mods)
    sub = long_df[cols].dropna().copy()
    valid_pairs = sub.groupby(subject_col)["visit"].nunique()
    valid_pairs = valid_pairs[valid_pairs == 2].index
    sub = sub[sub[subject_col].isin(valid_pairs)]
    modulator_denominators.append({
        "modulators": ",".join(mods),
        "n_modulators": len(mods),
        "n_subjects": int(sub[split_group_col].nunique()),
        "n_pairs": int(sub[subject_col].nunique()),
    })
modulator_denominator_table = pd.DataFrame(modulator_denominators)
modulator_denominator_table.to_csv(RESULTS_DIR / "interaction_modulator_denominators.csv", index=False)

full_n_subjects = int(modulator_denominator_table["n_subjects"].max())
full_n_pairs = int(modulator_denominator_table["n_pairs"].max())
modulator_denominator_table["full_denominator"] = (
    modulator_denominator_table["n_subjects"].eq(full_n_subjects)
    & modulator_denominator_table["n_pairs"].eq(full_n_pairs)
)
excluded_denominator = modulator_denominator_table.loc[~modulator_denominator_table["full_denominator"]].copy()
if excluded_denominator.empty:
    print("All modulator sets preserve the full denominator")
else:
    print("Excluding modulator sets that do not preserve the full 117-subject / 207-pair denominator")
    display(excluded_denominator)
modulator_sets = [tuple(str(x).split(",")) for x in modulator_denominator_table.loc[modulator_denominator_table["full_denominator"], "modulators"]]

print("Patient-adaptive interaction uses the SRM-selected 16 features")
display(pd.DataFrame([{
    "feature_set": "srm_elbow_selected_k16",
    "feature_count": len(selected_16_features),
    "features": ", ".join(selected_16_features),
}]))
print("Complete-case denominator by modulator set")
display(modulator_denominator_table)


Excluding modulator sets that do not preserve the full 117-subject / 207-pair denominator


,modulators,n_modulators,n_subjects,n_pairs,full_denominator
3,gaa_2,1,114,202,False
6,"age,gaa_2",2,114,202,False
8,"disease_duration,gaa_2",2,114,202,False
9,"gaa_1,gaa_2",2,114,202,False
11,"age,disease_duration,gaa_2",3,114,202,False


Patient-adaptive interaction uses the SRM-selected 16 features


,feature_set,feature_count,features
0,srm_elbow_selected_k16,16,"Lateral_Ventricle, Cerebellum_Cortex_CerebNet,..."


Complete-case denominator by modulator set


,modulators,n_modulators,n_subjects,n_pairs,full_denominator
0,age,1,117,207,True
1,disease_duration,1,117,207,True
2,gaa_1,1,117,207,True
3,gaa_2,1,114,202,False
4,"age,disease_duration",2,117,207,True
5,"age,gaa_1",2,117,207,True
6,"age,gaa_2",2,114,202,False
7,"disease_duration,gaa_1",2,117,207,True
8,"disease_duration,gaa_2",2,114,202,False
9,"gaa_1,gaa_2",2,114,202,False


## 3. Final Patient-Adaptive OOF Evaluation

The chosen configuration is rerun with confidence intervals enabled. These are genuine out-of-fold scores from grouped participant splits.


In [5]:

adaptive_search = evaluate_patient_adaptive_candidates(
    long_df,
    feature_sets_for_run,
    modulator_sets,
    subject_col=subject_col,
    visit_col="visit",
    split_group_col=split_group_col,
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    n_boot=N_BOOT,
    alpha_grid=(0.003, 0.01, 0.03, 0.1, 0.3, 0.7, 1.0, 1.3, 1.7, 2.0, 3.0, 5.0),
    l1_ratio_grid=(0.0,),
    z_clip_grid=(None, 4.0, 2.75, 2.0),
    inner_cv_splits=3,
)
adaptive_summary = adaptive_search["summary"]
if adaptive_summary.empty:
    raise RuntimeError("No patient-adaptive candidates were evaluated")

# Keep the best z-clipping/tuning row for each modulator combination, then rank
# by held-out annual d_z and interval balance.
modulator_screen = (
    adaptive_summary
    .sort_values(
        ["test_score", "test_annual_interval_gap", "modulator_count", "z_clip"],
        ascending=[False, True, True, True],
        kind="mergesort",
    )
    .groupby("modulators", as_index=False, sort=False)
    .head(1)
    .reset_index(drop=True)
)
modulator_screen["rank"] = np.arange(1, len(modulator_screen) + 1)
modulator_screen = modulator_screen.rename(columns={"modulator_count": "n_modulators"})
modulator_screen.to_csv(RESULTS_DIR / "interaction_modulator_combination_summary.csv", index=False)
adaptive_summary.to_csv(RESULTS_DIR / "interaction_adaptive_candidate_summary.csv", index=False)

print("Patient-adaptive tuning after locking to SRM-selected k=16 features")
display(adaptive_summary[[
    "rank", "feature_set", "feature_count", "modulators", "modulator_count", "z_clip",
    "validation_score", "test_score", "validation_minus_test", "test_dz_v1_v2", "test_dz_v2_v3",
    "test_annual_interval_gap", "test_p_progression", "pooled_pair_d_z", "n_subjects", "n_pairs",
]].head(25))

print("Best row per modulator combination")
display(modulator_screen[[
    "rank", "modulators", "n_modulators", "z_clip", "validation_score", "test_score",
    "test_dz_v1_v2", "test_dz_v2_v3", "test_annual_interval_gap", "test_p_progression",
    "pooled_pair_d_z", "n_subjects", "n_pairs",
]].head(12))

best_adaptive = adaptive_summary.iloc[0]
print("Best patient-adaptive configuration")
display(pd.DataFrame([best_adaptive]))


Patient-adaptive tuning after locking to SRM-selected k=16 features


,rank,feature_set,feature_count,modulators,modulator_count,z_clip,validation_score,test_score,validation_minus_test,test_dz_v1_v2,test_dz_v2_v3,test_annual_interval_gap,test_p_progression,pooled_pair_d_z,n_subjects,n_pairs
0,1,srm_elbow_selected_k16,16,gaa_1,1,NaN,0.681474,0.741727,-0.060252,0.853232,0.630221,0.223011,0.784933,0.730837,117,207
1,2,srm_elbow_selected_k16,16,gaa_1,1,4.00,0.675239,0.740182,-0.064943,0.856136,0.624228,0.231908,0.770623,0.728244,117,207
2,3,srm_elbow_selected_k16,16,gaa_1,1,2.75,0.668750,0.722598,-0.053848,0.835124,0.610071,0.225052,0.770202,0.712749,117,207
3,4,srm_elbow_selected_k16,16,age,1,NaN,0.708854,0.679303,0.029551,0.778681,0.579926,0.198755,0.776936,0.660555,117,207
4,5,srm_elbow_selected_k16,16,gaa_1,1,2.00,0.644591,0.678955,-0.034364,0.773193,0.584718,0.188476,0.770202,0.676696,117,207
5,6,srm_elbow_selected_k16,16,age,1,4.00,0.709609,0.677141,0.032468,0.781043,0.573239,0.207804,0.786616,0.657288,117,207
6,7,srm_elbow_selected_k16,16,"age,gaa_1",2,NaN,0.652228,0.671928,-0.019700,0.707990,0.635865,0.072125,0.775673,0.667003,117,207
7,8,srm_elbow_selected_k16,16,age,1,2.75,0.698608,0.669042,0.029567,0.768055,0.570028,0.198027,0.786195,0.650356,117,207
8,9,srm_elbow_selected_k16,16,"age,disease_duration,gaa_1",3,NaN,0.572052,0.666518,-0.094466,0.765048,0.567988,0.197060,0.771465,0.656402,117,207
9,10,srm_elbow_selected_k16,16,"age,disease_duration,gaa_1",3,4.00,0.573310,0.662238,-0.088928,0.742161,0.582315,0.159846,0.776515,0.656215,117,207


Best row per modulator combination


,rank,modulators,n_modulators,z_clip,validation_score,test_score,test_dz_v1_v2,test_dz_v2_v3,test_annual_interval_gap,test_p_progression,pooled_pair_d_z,n_subjects,n_pairs
0,1,gaa_1,1,NaN,0.681474,0.741727,0.853232,0.630221,0.223011,0.784933,0.730837,117,207
1,2,age,1,NaN,0.708854,0.679303,0.778681,0.579926,0.198755,0.776936,0.660555,117,207
2,3,"age,gaa_1",2,NaN,0.652228,0.671928,0.707990,0.635865,0.072125,0.775673,0.667003,117,207
3,4,"age,disease_duration,gaa_1",3,NaN,0.572052,0.666518,0.765048,0.567988,0.197060,0.771465,0.656402,117,207
4,5,disease_duration,1,2.75,0.657457,0.657261,0.716562,0.597960,0.118602,0.805556,0.658501,117,207
5,6,"age,disease_duration",2,NaN,0.623616,0.650454,0.763827,0.537081,0.226746,0.765993,0.641876,117,207
6,7,"disease_duration,gaa_1",2,4.00,0.598478,0.622692,0.630742,0.614642,0.016100,0.776936,0.624550,117,207


Best patient-adaptive configuration


,trial_id,feature_set,feature_count,modulators,modulator_count,z_clip,validation_score,test_score,validation_minus_test,test_dz_v1_v2,...,test_p_progression,pooled_pair_d_z,n_subjects,n_pairs,cv_mode,cv_n_splits,alpha,l1_ratio,features,rank
0,9,srm_elbow_selected_k16,16,gaa_1,1,NaN,0.681474,0.741727,-0.060252,0.853232,...,0.784933,0.730837,117,207,group_kfold,5,0.003,0.0,"Lateral_Ventricle, Cerebellum_Cortex_CerebNet,...",1


## 4. Compare With Current Best Model

This table compares the selected Patient-Adaptive model with the current best SRM Global Linear model from `srm_composite.ipynb`. The comparison uses the same annual interval metrics from the optimization logs.


In [6]:

# Compare adaptive result against the strongest non-adaptive 70-panel SRM candidate.
best_base = panel_pool_summary.iloc[0]
comparison = pd.DataFrame([
    {
        "model": "SRM panel combination",
        "feature_set": best_base["panel_combo"],
        "feature_count": best_base["feature_count"],
        "modulators": "none",
        "validation_score": best_base["validation_score"],
        "test_score": best_base["test_score"],
        "validation_minus_test": best_base["validation_minus_test"],
        "test_dz_v1_v2": best_base["test_dz_v1_v2"],
        "test_dz_v2_v3": best_base["test_dz_v2_v3"],
        "test_annual_interval_gap": best_base["test_annual_interval_gap"],
        "test_p_progression": best_base["test_p_progression"],
    },
    {
        "model": "Patient-adaptive interaction",
        "feature_set": best_adaptive["feature_set"],
        "feature_count": best_adaptive["feature_count"],
        "modulators": best_adaptive["modulators"],
        "validation_score": best_adaptive["validation_score"],
        "test_score": best_adaptive["test_score"],
        "validation_minus_test": best_adaptive["validation_minus_test"],
        "test_dz_v1_v2": best_adaptive["test_dz_v1_v2"],
        "test_dz_v2_v3": best_adaptive["test_dz_v2_v3"],
        "test_annual_interval_gap": best_adaptive["test_annual_interval_gap"],
        "test_p_progression": best_adaptive["test_p_progression"],
    },
])
comparison["adaptive_minus_base_test_score"] = np.nan
comparison.loc[comparison["model"].eq("Patient-adaptive interaction"), "adaptive_minus_base_test_score"] = (
    float(best_adaptive["test_score"]) - float(best_base["test_score"])
)
comparison.to_csv(RESULTS_DIR / "interaction_vs_base_model_summary.csv", index=False)
print("Patient-adaptive vs best non-adaptive 70-panel SRM")
display(comparison)

best_trial = adaptive_search["results"][str(int(best_adaptive["trial_id"]))]
chosen_params = best_trial.get("chosen_params_df", pd.DataFrame()).copy()
if not chosen_params.empty:
    fold_scores = chosen_params.rename(columns={
        "outer_fold": "fold",
        "outer_train_n_subjects": "train_subjects",
        "outer_validation_n_subjects": "validation_subjects",
        "outer_train_n_pairs": "train_pairs",
        "outer_validation_n_pairs": "validation_pairs",
        "outer_validation_d": "validation_d",
    }).copy()
    fold_scores.to_csv(RESULTS_DIR / "interaction_outer_grouped_cv_fold_scores.csv", index=False)
    print("Best patient-adaptive fold-level validation results")
    display(fold_scores[[
        "fold", "train_subjects", "validation_subjects", "train_pairs", "validation_pairs",
        "validation_d", "outer_validation_dz_v1_v2", "outer_validation_dz_v2_v3",
        "outer_validation_annual_interval_gap", "outer_validation_p_progression", "modulators", "n_features",
    ]])

validation_test_review = validation_test_gap_table(
    "Patient-Adaptive selected-feature interaction",
    test_intervals=best_trial["intervals"],
    chosen_params=best_trial.get("chosen_params_df"),
)
performance_rows = assemble_performance_rows(
    "Patient-Adaptive selected-feature interaction",
    composite_intervals=best_trial["intervals"],
    validation_test=validation_test_review,
    cv_mode=f"subject-level grouped {CV_N_SPLITS}-fold",
    source="interaction_term.ipynb",
)
validation_test_review.to_csv(RESULTS_DIR / "interaction_validation_test_gap.csv", index=False)
performance_rows.to_csv(RESULTS_DIR / "patient_adaptive_performance_summary.csv", index=False)
print("Validation vs held-out/test overfit check")
display(validation_test_review)
print("Patient-adaptive model evaluation table with validation/test columns")
display(performance_rows)

# Fit one final interaction model on all denominator-aligned data to describe
# the selected model's effective feature directions at the cohort-mean modulator profile.
best_modulators = tuple(str(best_adaptive["modulators"]).split(","))
final_cols = [subject_col, "visit", split_group_col] + selected_16_features + list(best_modulators)
final_train = long_df[final_cols].dropna().copy()
valid_pairs = final_train.groupby(subject_col)["visit"].nunique()
valid_pairs = valid_pairs[valid_pairs == 2].index
final_train = final_train[final_train[subject_col].isin(valid_pairs)].copy()
config = Config(
    random_state=RANDOM_SEED,
    interaction_en_alpha=float(best_adaptive["alpha"]) if "alpha" in best_adaptive else 0.1,
    interaction_en_l1_ratio=float(best_adaptive["l1_ratio"]) if "l1_ratio" in best_adaptive else 0.0,
    interaction_tune_inner_cv=False,
    interaction_z_clip=None if pd.isna(best_adaptive.get("z_clip", np.nan)) else float(best_adaptive["z_clip"]),
)
final_model = InteractionLinearComposite(config=config)
final_model.fit(
    final_train[selected_16_features],
    final_train[list(best_modulators)],
    final_train[subject_col].values,
    final_train["visit"].values,
    cv_group_id=final_train[split_group_col].values,
)
z_profile = {m: float(final_train[m].mean()) for m in best_modulators}
effective_weights = final_model.imaging_weights(z_profile)
interaction_importance = pd.DataFrame({
    "feature": effective_weights.index,
    "interaction_effective_coefficient": effective_weights.values.astype(float),
})
interaction_importance["interaction_coefficient_sign"] = np.sign(
    interaction_importance["interaction_effective_coefficient"]
).astype(int)
interaction_importance["selected_modulators"] = ",".join(best_modulators)
interaction_importance["modulator_profile"] = "full cohort mean"
interaction_importance["train_n_subjects"] = int(final_train[split_group_col].nunique())
interaction_importance["train_n_pairs"] = int(final_train[subject_col].nunique())
interaction_importance["abs_interaction_effective_coefficient"] = interaction_importance[
    "interaction_effective_coefficient"
].abs()
interaction_importance = interaction_importance.sort_values(
    "abs_interaction_effective_coefficient", ascending=False, kind="mergesort"
).reset_index(drop=True)
interaction_importance.insert(0, "rank", np.arange(1, len(interaction_importance) + 1))
interaction_importance.to_csv(RESULTS_DIR / "interaction_final_model_feature_importance.csv", index=False)
print("Interaction feature importance at cohort-mean modulator profile")
display(interaction_importance)


Patient-adaptive vs best non-adaptive 70-panel SRM


,model,feature_set,feature_count,modulators,validation_score,test_score,validation_minus_test,test_dz_v1_v2,test_dz_v2_v3,test_annual_interval_gap,test_p_progression,adaptive_minus_base_test_score
0,SRM panel combination,brain_structural+brain_diffusion_rd,40,none,0.665371,0.808814,-0.143443,1.000076,0.617553,0.382523,0.813131,NaN
1,Patient-adaptive interaction,srm_elbow_selected_k16,16,gaa_1,0.681474,0.741727,-0.060252,0.853232,0.630221,0.223011,0.784933,-0.067088


Best patient-adaptive fold-level validation results


,fold,train_subjects,validation_subjects,train_pairs,validation_pairs,validation_d,outer_validation_dz_v1_v2,outer_validation_dz_v2_v3,outer_validation_annual_interval_gap,outer_validation_p_progression,modulators,n_features
0,1,93,24,162,45,0.841271,1.211247,0.471296,0.739951,0.791667,gaa_1,16
1,2,93,24,159,48,0.713620,0.790410,0.636830,0.153580,0.750000,gaa_1,16
2,3,94,23,169,38,0.712094,0.908315,0.515873,0.392441,0.728291,gaa_1,16
3,4,94,23,167,40,0.612880,0.538834,0.686926,0.148093,0.778195,gaa_1,16
4,5,94,23,171,36,1.073580,0.983495,1.163664,0.180170,0.888889,gaa_1,16


Validation vs held-out/test overfit check


,model,question,metric,validation_score,test_score,validation_minus_test,overfit_check,evidence
0,Patient-Adaptive selected-feature interaction,12-month sensitivity V1->V2,validation d_z vs held-out/test d_z,0.792058,0.853232,-0.061174,aligned,inner grouped CV compared with outer held-out ...
1,Patient-Adaptive selected-feature interaction,12-month sensitivity V2->V3,validation d_z vs held-out/test d_z,0.570891,0.630221,-0.059331,aligned,inner grouped CV compared with outer held-out ...
2,Patient-Adaptive selected-feature interaction,12-month pooled annual sensitivity,validation mean annual d_z vs held-out/test d_z,0.681474,NaN,NaN,,inner grouped CV compared with outer held-out ...


Patient-adaptive model evaluation table with validation/test columns


,model,question,metric,role,value,n,status,evidence,cv_mode,source,validation_score,test_score,validation_minus_test,overfit_check
0,Patient-Adaptive selected-feature interaction,12-month sensitivity V1->V2,"V1->V2 paired d_z, CI, N, P(delta>0)",Primary,"0.8532320940154092 [0.6423654259893941, 1.1234...",108.0,computed,composite V1->V2 OOF annual interval,subject-level grouped 5-fold,interaction_term.ipynb,0.792058,0.853232,-0.061174,aligned
1,Patient-Adaptive selected-feature interaction,12-month sensitivity V2->V3,"V2->V3 paired d_z, CI, N, P(delta>0)",Primary temporal replication,"0.6302211241838507 [0.43854855558560657, 0.827...",99.0,computed,composite V2->V3 OOF annual interval,subject-level grouped 5-fold,interaction_term.ipynb,0.570891,0.630221,-0.059331,aligned
2,Patient-Adaptive selected-feature interaction,12-month pooled annual sensitivity,"Pooled V1->V2 + V2->V3 paired d_z, CI, N, P(de...",Pooled annual diagnostic,NaN,NaN,missing,pooled annual V1->V2 + V2->V3,subject-level grouped 5-fold,interaction_term.ipynb,0.681474,NaN,NaN,
3,Patient-Adaptive selected-feature interaction,24-month cumulative sensitivity,V1->V3 paired d_z,Secondary,NaN,NaN,missing,composite V1->V3 cumulative,subject-level grouped 5-fold,interaction_term.ipynb,NaN,NaN,NaN,
4,Patient-Adaptive selected-feature interaction,Direction consistency,P(delta > 0),Secondary,0.8425925925925926; 0.7272727272727273,108.0,computed,annual V1->V2 and V2->V3 P(delta>0),subject-level grouped 5-fold,interaction_term.ipynb,NaN,NaN,NaN,
5,Patient-Adaptive selected-feature interaction,Robustness,bootstrap CI for d_z,Primary uncertainty,"V1->V2 [0.6423654259893941, 1.123420717650602]...",108.0,computed,annual interval bootstrap CI,subject-level grouped 5-fold,interaction_term.ipynb,NaN,NaN,NaN,
6,Patient-Adaptive selected-feature interaction,Better than clinical scale?,d_z composite vs FARS/SARA,RQ1,nan vs nan,NaN,missing_reference,clinical interval benchmark,subject-level grouped 5-fold,interaction_term.ipynb,NaN,NaN,NaN,
7,Patient-Adaptive selected-feature interaction,Better than MRI alone?,vs strongest individual MRI feature,RQ1,nan vs nan: nan,NaN,missing_reference,strongest single MRI feature,subject-level grouped 5-fold,interaction_term.ipynb,NaN,NaN,NaN,
8,Patient-Adaptive selected-feature interaction,Disease specific?,FRDA vs control change,Specificity,NaN,NaN,missing,FRDA vs control change,subject-level grouped 5-fold,interaction_term.ipynb,NaN,NaN,NaN,
9,Patient-Adaptive selected-feature interaction,Clinically meaningful?,Spearman Z vs FARS/SARA,RQ3,NaN,NaN,missing,cross-sectional Spearman,subject-level grouped 5-fold,interaction_term.ipynb,NaN,NaN,NaN,


Interaction feature importance at cohort-mean modulator profile


,rank,feature,interaction_effective_coefficient,interaction_coefficient_sign,selected_modulators,modulator_profile,train_n_subjects,train_n_pairs,abs_interaction_effective_coefficient
0,1,Cerebellum_Cortex_CerebNet,-2.465629,-1,gaa_1,full cohort mean,117,207,2.465629
1,2,Midbrain,-1.295954,-1,gaa_1,full cohort mean,117,207,1.295954
2,3,Medulla,1.156116,1,gaa_1,full cohort mean,117,207,1.156116
3,4,Putamen,1.054647,1,gaa_1,full cohort mean,117,207,1.054647
4,5,TotalBrainGMVol_nocereb,-0.850934,-1,gaa_1,full cohort mean,117,207,0.850934
5,6,Cerebellum_WM_CerebNet,-0.841034,-1,gaa_1,full cohort mean,117,207,0.841034
6,7,Lateral_Ventricle,0.821350,1,gaa_1,full cohort mean,117,207,0.821350
7,8,FA_PTR,-0.641773,-1,gaa_1,full cohort mean,117,207,0.641773
8,9,Caudate,-0.620507,-1,gaa_1,full cohort mean,117,207,0.620507
9,10,Pons,0.605644,1,gaa_1,full cohort mean,117,207,0.605644
